# 04. Synthetic dataset EDA

synthetic stage dataset을 확인한다. public stage보다 규모가 작고 panel 형태에 가깝게 생성된
이미지이므로 corner 분포와 이미지 특성이 어떻게 다른지 함께 살펴본다.

## 0. 환경 설정

In [ ]:
import os
import sys

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(PROJECT_ROOT)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from src.data.dataset import CornerDataset
from src.core.factory import get_transform, get_dataset, get_dataloader
from src.utils.plot import show_samples, show_corner_scatter

In [ ]:
CSV_PATH = [
    os.path.join(PROJECT_ROOT, "data", "synthetic", "fake", "gt_corners.csv"),
    os.path.join(PROJECT_ROOT, "data", "synthetic", "similar", "gt_corners.csv"),
    os.path.join(PROJECT_ROOT, "data", "synthetic", "augmented", "gt_corners.csv"),
]
IMAGE_SIZE = 224
BATCH_SIZE = 4
SEED = 42

NUM_SHOW = 10        # number of sample images to display
NUM_SCATTER = 2000   # number of corner sets used for the distribution plot

for path in CSV_PATH:
    print(os.path.exists(path), path)

## 1. Dataset 로딩

`CornerDataset`은 CSV의 `image_dir`, `image_name`과 `x1..y4` 열을 읽어 (image, corners) 쌍을 만든다.
synthetic stage는 fake, similar, augmented 세 하위 dataset으로 구성되며 각각 `data/make_fake_images.py`,
`data/make_similar_images.py`, `data/make_augmented_images.py`로 생성한다. CSV가 없으면
`notebooks/00-data-generation.ipynb`를 먼저 실행해야 한다.

In [ ]:
dataset = CornerDataset(CSV_PATH)
print("total samples:", len(dataset))

image_path, corners = dataset.samples[0]
print("image path:", image_path)
print("corners shape:", corners.shape, corners.dtype)
print(corners)

In [ ]:
image, corners = dataset[0]
print("image:", type(image).__name__, tuple(image.shape), image.dtype)
print("corners:", tuple(corners.shape), corners.dtype)

## 2. Train / valid / test split

`get_dataset`은 `split_ratio=0.6` 기준으로 60:20:20 split을 내부에서 수행한다.
synthetic dataset은 규모가 작아 valid와 test split의 sample 수도 적다.

In [ ]:
train_dataset = get_dataset("train", CSV_PATH, image_size=IMAGE_SIZE, seed=SEED)
valid_dataset = get_dataset("valid", CSV_PATH, image_size=IMAGE_SIZE, seed=SEED)
test_dataset = get_dataset("test", CSV_PATH, image_size=IMAGE_SIZE, seed=SEED)

total = len(train_dataset) + len(valid_dataset) + len(test_dataset)
for name, subset in [("train", train_dataset), ("valid", valid_dataset), ("test", test_dataset)]:
    print("%-6s %6d (%.1f%%)" % (name, len(subset), 100.0 * len(subset) / total))
print("%-6s %6d" % ("total", total))

## 3. 샘플 이미지 확인

train split에서 일부를 뽑아 corner overlay와 함께 표시한다. transform이 정규화를 포함하므로
`denormalize=True`로 원래 픽셀 범위를 복원해서 그린다.

In [ ]:
samples = [train_dataset.subset(NUM_SHOW, seed=SEED)[i] for i in range(NUM_SHOW)]
images = [s[0] for s in samples]
corners = [s[1] for s in samples]

show_samples(images, corners, ncols=5, title="synthetic train samples", denormalize=True)

In [ ]:
samples = [test_dataset.subset(NUM_SHOW, seed=SEED)[i] for i in range(NUM_SHOW)]
images = [s[0] for s in samples]
corners = [s[1] for s in samples]

show_samples(images, corners, ncols=5, title="synthetic test samples", denormalize=True)

## 4. Corner 좌표 분포

CSV의 원본 corner 좌표를 그대로 모아 분포를 확인한다. 이미지를 읽지 않으므로 전체 sample을
대상으로 해도 빠르다. y축은 이미지 좌표계에 맞춰 위에서 아래로 증가하도록 뒤집는다.

In [ ]:
all_corners = np.stack([c for _, c in dataset.samples])
print("all corners shape:", all_corners.shape)

rng = np.random.default_rng(SEED)
if len(all_corners) > NUM_SCATTER:
    idx = rng.choice(len(all_corners), NUM_SCATTER, replace=False)
    sampled = all_corners[idx]
else:
    sampled = all_corners

show_corner_scatter(sampled, title="synthetic corner distribution")

In [ ]:
labels = ["TL", "TR", "BR", "BL"]
print("%-4s %-22s %-22s" % ("", "x (min/mean/max)", "y (min/mean/max)"))
for j, label in enumerate(labels):
    xs, ys = all_corners[:, j, 0], all_corners[:, j, 1]
    print("%-4s %6.3f %6.3f %6.3f   %6.3f %6.3f %6.3f"
          % (label, xs.min(), xs.mean(), xs.max(), ys.min(), ys.mean(), ys.max()))

## 5. Transform 적용 확인

train transform은 flip, rotation, color jitter, blur를 포함하고 test transform은 resize와
정규화만 수행한다. 같은 이미지에 두 transform을 적용해 tensor 속성을 비교한다.

In [ ]:
from PIL import Image

image_path, raw_corners = dataset.samples[0]
raw_image = Image.open(image_path).convert("RGB")
print("original size:", raw_image.size)

for split in ["train", "test"]:
    transform = get_transform(split, image_size=IMAGE_SIZE)
    image, corners = transform(raw_image, raw_corners.copy())
    print("%-6s shape=%s dtype=%s min=%.3f max=%.3f"
          % (split, tuple(image.shape), image.dtype, image.min(), image.max()))

In [ ]:
transform = get_transform("train", image_size=IMAGE_SIZE)
augmented = [transform(raw_image, raw_corners.copy()) for _ in range(NUM_SHOW)]

show_samples([a[0] for a in augmented], [a[1] for a in augmented], ncols=5,
             title="train transform applied to one image", denormalize=True)

## 6. Dataloader 확인

`get_dataloader`는 split별 dataset을 batch 단위로 묶는다. `num_samples`를 주면 해당 split에서
무작위로 일부만 뽑아 빠르게 확인할 수 있다.

In [ ]:
train_loader = get_dataloader("train", CSV_PATH, image_size=IMAGE_SIZE, seed=SEED,
                              batch_size=BATCH_SIZE, num_workers=0, num_samples=100)
print("batches:", len(train_loader), "batch size:", BATCH_SIZE)

images, corners = next(iter(train_loader))
print("images:", tuple(images.shape), images.dtype)
print("corners:", tuple(corners.shape), corners.dtype)

In [ ]:
show_samples(list(images), list(corners), ncols=BATCH_SIZE,
             title="one train batch", denormalize=True)